# KGSAGE → ADKGD: full pipeline from scratch

This notebook trains everything from the beginning and produces the downstream
**2×2 anomaly-detection matrix** on FB15K-237 and WN18RR.

**Pipeline (top to bottom):**

```
data/<KG>/{train,valid,test}.txt
        │
        ▼  ONE command  (python -m kgsage.gan.train)
  ┌─────────────────────────────────────────────┐
  │ Phase 1  RGCN warm-up  → frozen context E'   │   (needs torch_geometric)
  │ Phase 2  dual-discriminator GAN game         │
  └─────────────────────────────────────────────┘
        │  final-epoch checkpoint (run_<KG>_s0.pt)
        ▼
  knockout J@10   (sanity: is the generator anchor-specific?)
        │
        ▼  ADKGD detector, 4 cells per KG
  2×2 matrix:  train-neg {random, gan} × test-anomaly {random, gan}
        │
        ▼
  aggregate → Precision@K / Recall@K / AUC / AUPRC tables
```

**Two things to know before running:**

1. **RGCN and the GAN are one command.** The RGCN warm-up (Phase 1, produces the
   frozen table E') and the adversarial game (Phase 2) run back-to-back *inside*
   `python -m kgsage.gan.train`; E' is handed to the generator in-process. There
   is no separate "train RGCN" CLI step — you'll see both phases in that cell's log.

2. **Every snapshot is kept, and §4 selects the best one.** Anchor-specificity
   peaks a few epochs after the α-ramp and then *erodes*, so the final epoch is
   usually **not** the best generator — an FB run that kept only the last epoch
   scored knockout J@10 **0.862** (collapsed) against **0.582** for a selected
   one. §4 scores every snapshot and feeds the winner to ADKGD. Set
   `SNAPSHOT_EVERY = 0` to go back to final-epoch-only.

**Runtime / hardware.** The generator training is light (minutes on CPU, faster on
GPU; PyG required). The **ADKGD 2×2 matrix is the heavy part** — each cell trains
the detector (~10+ min/epoch for FB on CPU). GPU strongly recommended; on the
HPC, prefer `experiments/slurm/exp_cell.slurm` for the matrix. This notebook runs
it inline too (slower).

## Bootstrap — only for a fresh machine / Colab

If you are running this notebook from **inside an already-cloned ADKGD repo**
(the normal case), the next cell is a **no-op** — skip straight to §0.

If you are on a **fresh environment** (Google Colab, a new HPC node, a clean
machine), the cells below clone the repo, **check out the `dev_gan_2` branch**
(the KGSAGE work is there, not on `main`), and `cd` in. FB15K-237 and WN18RR
data are committed in the repo, so nothing else needs downloading. The optional
last cell installs Python dependencies (needed on Colab).

In [ ]:
# --- Idempotent bootstrap: clone the repo only if we're not already inside it ---
import os, subprocess
from pathlib import Path

REPO_URL     = "https://github.com/tharindu-wj/ADKGD.git"  # private repo? use a token:
                                                           # https://<TOKEN>@github.com/tharindu-wj/ADKGD.git
REPO_DIRNAME = "ADKGD"

def _is_repo(p):
    p = Path(p)
    return (p / "Our_TopK%_RankingList.py").exists() and (p / "experiments").is_dir()

if any(_is_repo(d) for d in [Path.cwd(), *Path.cwd().parents]):
    print("Already inside the ADKGD repo — no clone needed.")
elif _is_repo(REPO_DIRNAME):
    os.chdir(REPO_DIRNAME); print("Repo already cloned; cd ->", Path.cwd())
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIRNAME], check=True)
    os.chdir(REPO_DIRNAME); print("Cloned + cd ->", Path.cwd())


In [ ]:
# --- Check out the working branch and pull the latest commits ---
# Safe to re-run: on a fresh clone this checks out dev_gan_2; on an existing
# clone it also fast-forwards to the newest commits, so you do NOT need to
# re-clone between sessions.
import subprocess

BRANCH = "dev_gan_2"


def git(*args, check=True):
    p = subprocess.run(["git", *args], capture_output=True, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{p.stderr.strip()}")
    return p.stdout.strip()


git("fetch", "origin", BRANCH)

# A previous session may have patched train.py (the tuning cell below edits it).
# Those edits would block a pull, and we do not want to keep them anyway --
# the tuning cell re-applies them after the pull.
dirty = git("status", "--porcelain", "--untracked-files=no")
if dirty:
    print("discarding local edits to tracked files before pulling:")
    print(dirty)
    git("checkout", "--", ".")

git("checkout", BRANCH)
git("merge", "--ff-only", f"origin/{BRANCH}")

current = git("rev-parse", "--abbrev-ref", "HEAD")
assert current == BRANCH, f"expected branch {BRANCH}, but on {current!r}"
print("on branch:", current, "@", git("log", "-1", "--oneline"))


In [ ]:
# --- Optional: install Python dependencies (fresh env / Colab only) ---
INSTALL_DEPS = False   # set True on a machine that doesn't already have the stack

if INSTALL_DEPS:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "scipy", "scikit-learn", "matplotlib", "pandas", "networkx"], check=True)
    # torch + torch_geometric (the RGCN warm-up needs PyG). On Colab torch is
    # preinstalled; PyG install can be CUDA/version-specific — see pyg.org if this
    # generic install fails.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch", "torch_geometric"], check=True)
    print("dependencies installed")
else:
    print("skipped (set INSTALL_DEPS = True on a fresh environment)")


## 0 · Environment & setup

In [ ]:
import os, sys, subprocess, json, shutil, time
from pathlib import Path

# --- locate the repo root (dir with Our_TopK%_RankingList.py + experiments/ + data/) ---
_here = Path.cwd()
REPO = None
for cand in [_here, *_here.parents]:
    if (cand / "Our_TopK%_RankingList.py").exists() and (cand / "experiments").is_dir():
        REPO = cand
        break
assert REPO is not None, "Could not find the repo root — open this notebook from inside the ADKGD repo."
os.chdir(REPO)
print("Repo root :", REPO)

# --- interpreter = this kernel's python (so kgsage's env is reused) ---
PY = sys.executable
print("Python    :", PY)

# --- PYTHONPATH so `python -m kgsage.gan.train` resolves; CPU-safety guards ---
os.environ["PYTHONPATH"] = str(REPO / "experiments") + os.pathsep + os.environ.get("PYTHONPATH", "")
# These guards prevent PyTorch CPU segfaults on local Windows. On an HPC GPU/CPU
# node, RAISE them to your allocated cores for speed (e.g. os.environ["OMP_NUM_THREADS"]="16").
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

def run(cmd):
    """Run a command from the repo root, stream its output live, raise on failure."""
    cmd = [str(c) for c in cmd]
    print("\n$ " + " ".join(cmd), flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, env=os.environ.copy(), cwd=str(REPO),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:          # stream child output INTO this cell (visible in Colab)
        print(line, end="", flush=True)
    proc.wait()
    dt = time.time() - t0
    print(f"[exit {proc.returncode} · {dt/60:.1f} min]", flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f"command failed with code {proc.returncode} (see traceback above)")


In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch    :", torch.__version__, "| device:", DEVICE)
try:
    import torch_geometric
    print("PyG      :", torch_geometric.__version__, "(required for the RGCN warm-up, Phase 1)")
except Exception as e:
    print("!! torch_geometric MISSING — the RGCN warm-up (Phase 1) REQUIRES it.")
    print("   FIX: set INSTALL_DEPS = True in the deps cell above and run it, then re-run from here.")
    print("   (", type(e).__name__, ")")


## 1 · Configuration

In [ ]:
SEED           = 0
KGSAGE_EPOCHS  = 8        # game epochs. Keep runs short: the
                         # anchor-specificity peak comes a few epochs after the alpha ramp
SNAPSHOT_EVERY  = 1      # save a checkpoint every N game epochs, then pick the best by
                         # knockout J@10. Set 0 to keep only the final epoch -- but note
                         # the FB run that did so scored 0.862 (collapsed) because
                         # anchor-specificity had already eroded by the last epoch.

# The one dial that decides how hard the corruptions are: the PI set-point for the
# fraction of the generator's picks that the training graph corroborates. It must sit
# INSIDE the range the dataset can actually reach, or the controller saturates.
#   FB15K-237 measured: alpha=0 -> 0.64, alpha=ALPHA_MAX -> 0.26. The shipped default
#   0.13 is BELOW that floor, which is why alpha railed at 10 and the generator
#   collapsed onto one universal contradiction. 0.30 sits inside the band.
#   Set to None to leave the checked-in default untouched.
CORROBORATION_TARGET = 0.30

ADKGD_MAX_EPOCH = 1      # ADKGD detector epochs per matrix cell (paper default = 1)
ANOMALY_RATIO   = 0.05   # fraction of injected anomalies (paper default)

# tag maps the data-folder name to the checkpoint tag used elsewhere in the repo
DATASETS = {
    "FB15K-237": {"data": "data/FB15K-237", "tag": "fb15k237", "knockout_relations": []},
    "WN18RR":    {"data": "data/WN18RR",    "tag": "wn18rr",
                  "knockout_relations": ["_hypernym", "_derivationally_related_form",
                                         "_member_meronym", "_has_part"]},
}
CKPT_DIR = REPO / "experiments" / "kgsage" / "outputs" / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# Which datasets to run in this notebook (edit to just one while iterating):
ACTIVE = ["FB15K-237", "WN18RR"]
print("Active datasets:", ACTIVE)


### 1b · Apply the tuning

Rewrites `CORROBORATION_TARGET` in `experiments/kgsage/gan/train.py`. Re-runnable
and self-verifying: it asserts exactly one line changed and prints the result. The
pull cell above resets the file first, so this never stacks up.

In [ ]:
import re, pathlib

if CORROBORATION_TARGET is None:
    print("skipped -- using the value checked into train.py")
else:
    p = pathlib.Path("experiments/kgsage/gan/train.py")
    text = p.read_text(encoding="utf-8")
    patched, n = re.subn(r"^(CORROBORATION_TARGET\s*=\s*)[\d.]+",
                         rf"\g<1>{CORROBORATION_TARGET}", text, flags=re.M)
    assert n == 1, f"expected exactly 1 replacement, got {n} -- check the constant name"
    p.write_text(patched, encoding="utf-8")
    print([l for l in patched.splitlines() if l.startswith("CORROBORATION_TARGET")][0])


## 2 · Verify data + plumbing (quick smoke)

In [ ]:
# The data dirs only need train/valid/test.txt — ADKGD initialises embeddings
# randomly (no entity2vec.txt required), and KGSAGE reads the same three files.
for ds in ACTIVE:
    for f in ("train.txt", "valid.txt", "test.txt"):
        p = REPO / DATASETS[ds]["data"] / f
        assert p.exists(), f"MISSING: {p}"
    print(f"OK: {ds} data present")

# package + bridge import checks (fast)
run([PY, "experiments/kgsage/smoke_test.py"])
run([PY, "experiments/kgsage_bridge/smoke_test.py"])


## 3 · Train the KGSAGE generator  (Phase 1 RGCN warm-up → Phase 2 GAN game)

One command per dataset. Watch the log and you will see the phases go by:

- `warmup 1/10 … 10/10` — **Phase 1**: the RGCN context encoder trained against a
  throwaway DistMult decoder, then the context table E' is frozen.
- `Neighbourhood discriminator pretraining` then the `dreal-pre` lines —
  **Phase 2a**: each discriminator is trained on its own before the game.
  (`dmatch` and `dreal-pre` are frozen log prefixes kept so older runs stay
  comparable; they mean the neighbourhood and plausibility discriminators.)
- `Dual-discriminator: N epochs …` — **Phase 2b**: the game itself.

A checkpoint is written every `SNAPSHOT_EVERY` game epochs as
`run_<KG>_s0.epNN.pt`, plus the final `run_<KG>_s0.pt`. §4 picks the best.

In [ ]:
# --- FB15K-237 ---
ds = "FB15K-237"
if ds in ACTIVE:
    cfg = DATASETS[ds]
    fb_ckpt = CKPT_DIR / f"run_{cfg['tag']}_s{SEED}.pt"
    run([PY, "-m", "kgsage.gan.train",
         "--data", cfg["data"],
         "--out", str(fb_ckpt),
         "--epochs", str(KGSAGE_EPOCHS),
         "--snapshot_every", str(SNAPSHOT_EVERY),
         "--seed", str(SEED),
         "--device", DEVICE])
    print("saved:", fb_ckpt)


In [ ]:
# --- WN18RR ---
ds = "WN18RR"
if ds in ACTIVE:
    cfg = DATASETS[ds]
    wn_ckpt = CKPT_DIR / f"run_{cfg['tag']}_s{SEED}.pt"
    run([PY, "-m", "kgsage.gan.train",
         "--data", cfg["data"],
         "--out", str(wn_ckpt),
         "--epochs", str(KGSAGE_EPOCHS),
         "--snapshot_every", str(SNAPSHOT_EVERY),
         "--seed", str(SEED),
         "--device", DEVICE])
    print("saved:", wn_ckpt)


## 4 · Sanity-check the generators (knockout J@10)

Lower J@10 = more anchor-aware (deleting the anchor's neighbourhood changes the
top-10). Reference from the locked artifacts: WN18RR ≈ 0.08 (strong), FB15K-237
≈ 0.58 (partial — collapses on small categorical relations). Since snapshots are
off, this reports the **final** epoch only.

In [ ]:
# Score EVERY snapshot and keep the most anchor-specific one (lowest mean
# knockout J@10). Anchor-specificity peaks a few epochs after the alpha ramp and
# then erodes, so the final epoch is usually NOT the best checkpoint.
import json, re

GAN_CKPT = {}   # dataset -> the snapshot we will feed to ADKGD

for ds in ACTIVE:
    cfg = DATASETS[ds]
    final = CKPT_DIR / f"run_{cfg['tag']}_s{SEED}.pt"
    snaps = sorted(CKPT_DIR.glob(f"run_{cfg['tag']}_s{SEED}.ep*.pt"))
    candidates = snaps + [final] if snaps else [final]
    rel_args = (["--relations", *cfg["knockout_relations"]] if cfg["knockout_relations"] else [])

    print(f"\n{'='*70}\n{ds}: scoring {len(candidates)} checkpoint(s)\n{'='*70}")
    scores = {}
    for ck in candidates:
        run([PY, "experiments/kgsage/cli/knockout_eval.py",
             "--ckpt", str(ck), "--data", cfg["data"], *rel_args])
        report = (REPO / "experiments/kgsage/outputs/eval/knockout" / f"{ck.stem}_knockout.json")
        if report.exists():
            scores[ck] = json.loads(report.read_text())["mean_knockout_j10"]

    if scores:
        best = min(scores, key=scores.get)
        print(f"\n--- {ds} knockout J@10 (lower = more anchor-specific) ---")
        for ck, v in sorted(scores.items(), key=lambda kv: kv[1]):
            print(f"  {v:.3f}  {ck.name}{'   <-- BEST' if ck == best else ''}")
        GAN_CKPT[ds] = best
    else:
        GAN_CKPT[ds] = final
        print("!! no JSON reports found; falling back to the final-epoch checkpoint")

print("\nGenerators for the matrix:")
for ds, p in GAN_CKPT.items():
    print(f"  {ds}: {p.name}")


## 5 · Optional — intrinsic corruption-quality artifacts (7.3 / 7.4)

Produces the corruptions CSV, ego graphs, and LLM/neighbourhood blocks for the
paper's qualitative evaluation. Cheap. Skip if you only want the downstream matrix.

In [ ]:
RUN_INTRINSIC = False   # set True to generate CSV + ego graphs + LLM blocks

if RUN_INTRINSIC:
    for ds in ACTIVE:
        cfg = DATASETS[ds]
        ckpt = GAN_CKPT[ds]
        # stage-1 corruptions CSV (shared by the LLM and ego evals)
        run([PY, "experiments/kgsage/cli/gen_corruptions_csv.py",
             "--ckpt", str(ckpt), "--data", cfg["data"],
             "--split", "test", "--per_rel", "4", "--seed", "7"])
        csv = REPO / "experiments/kgsage/outputs/eval/gen_corruptions" / f"{ds}_test_corruptions.csv"
        # ego graphs (7.4)
        run([PY, "experiments/kgsage/cli/ego_from_csv.py",
             "--csv", str(csv), "--data", cfg["data"], "--limit", "6"])
        # LLM triple blocks (7.3) + neighbourhood-contradiction blocks (7.4 semantic)
        run([PY, "experiments/kgsage/cli/format_for_llm.py", "--csv", str(csv)])
        run([PY, "experiments/kgsage/cli/gen_neighbourhood_context.py",
             "--csv", str(csv), "--data", cfg["data"]])
else:
    print("skipped (set RUN_INTRINSIC = True to enable)")


## 6 · ADKGD 2×2 downstream matrix

Four cells per dataset — the axes are **train negatives** × **test anomalies**,
each ∈ {`random` = rule-based baseline, `gan` = KGSAGE}:

| cell | train-neg | test-anom | meaning |
|---|---|---|---|
| baseline | random | random | ADKGD's original protocol |
| the gap | random | gan | baseline detector vs. hard KGSAGE anomalies |
| proposed | gan | gan | trained on KGSAGE, tested on KGSAGE |
| cross-check | gan | random | trained on KGSAGE, tested on easy anomalies |

Each cell runs the ADKGD detector **train + test** (`run_experiment.py`) using the
**frozen** generator — the generator is not retrained here. A per-cell
`checkpoints/<dataset>/…_run.json` is written for aggregation.

⚠️ **Arm-4 caveat (`gan × gan`):** train-gan × test-gan can leak / be circular
(≈5–7% inflation noted in earlier diagnosis). Report it with that caveat, or apply
the leakage fix before quoting it as the headline number.

⚠️ **Heavy step.** On CPU each FB cell can take 10+ min. For a real run prefer the
HPC GPU partition via `experiments/slurm/exp_cell.slurm`. Lower `ADKGD_MAX_EPOCH`
or run one dataset at a time while iterating.

In [ ]:
MATRIX = [("random", "random"), ("random", "gan"), ("gan", "random"), ("gan", "gan")]

def run_matrix(ds):
    cfg = DATASETS[ds]
    gan = GAN_CKPT[ds]
    print(f"\n########## ADKGD 2x2 matrix — {ds} ##########")
    for neg, test in MATRIX:
        cmd = [PY, "experiments/run_experiment.py",
               "--dataset", ds,
               "--neg_source", neg,
               "--test_anomaly_source", test,
               "--seed", str(SEED),
               "--max_epoch", str(ADKGD_MAX_EPOCH),
               "--anomaly_ratio", str(ANOMALY_RATIO)]
        if neg == "gan" or test == "gan":
            cmd += ["--gan_path", str(gan)]
        run(cmd)


In [ ]:
# --- FB15K-237 matrix (4 cells) ---
if "FB15K-237" in ACTIVE:
    run_matrix("FB15K-237")


In [ ]:
# --- WN18RR matrix (4 cells) ---
if "WN18RR" in ACTIVE:
    run_matrix("WN18RR")


## 7 · Aggregate & view the results

In [ ]:
# mean±std tables per matrix cell (grouped by dataset × train-neg × test-anom)
for ds in ACTIVE:
    run([PY, "experiments/aggregate_results.py", "--dataset", ds])


In [ ]:
# Pretty 2x2 view straight from the per-run JSON records
import json
import pandas as pd

rows = []
for jp in sorted((REPO / "checkpoints").glob("**/*_run.json")):
    r = json.loads(jp.read_text(encoding="utf-8"))
    rows.append({
        "dataset": r["dataset"], "train_neg": r["neg_source"], "test_anom": r["test_anomaly_source"],
        "AUC": r.get("auc"), "AUPRC": r.get("auprc"),
        **{f"P@{k}": v for k, v in r.get("precision_at", {}).items()},
    })

if rows:
    df = pd.DataFrame(rows).sort_values(["dataset", "train_neg", "test_anom"]).reset_index(drop=True)
    display(df)
    for ds in df["dataset"].unique():
        print(f"\n{ds} — AUC  (rows = train-neg, cols = test-anom):")
        display(df[df.dataset == ds].pivot(index="train_neg", columns="test_anom", values="AUC"))
else:
    print("No *_run.json found yet — run the matrix cells above first.")


## Notes

- **Artifacts.** Generators: `experiments/kgsage/outputs/checkpoints/run_<tag>_s0.pt`.
  ADKGD per-run records + logs: `checkpoints/<dataset>/`.
- **Locked artifacts untouched.** This notebook feeds ADKGD the freshly-trained
  checkpoints; it does not overwrite your validated `generator_fb15k237.pt` /
  `generator_wn18rr.pt`.
- **Snapshot selection.** §4 scores every `.epNN.pt` by knockout J@10 and feeds
  the lowest (most anchor-specific) one to ADKGD. `SNAPSHOT_EVERY = 0` disables
  this and keeps only the final epoch.
- **Tuning.** `CORROBORATION_TARGET` (§1) must sit inside the range the dataset
  can reach, or the PI controller saturates at `ALPHA_MAX` and the generator
  collapses. FB15K-237 measures alpha=0 -> 0.64 and alpha=max -> 0.26, so the
  shipped 0.13 is below the floor; 0.30 is inside it.
- **HPC.** Generator: `DATASET=<fb15k237|wn18rr> sbatch experiments/kgsage/slurm/train.slurm`
  (GPU) or `train_cpu.slurm` (CPU). Matrix: `NEG_SOURCE=.. TEST_SOURCE=.. DATASET=..
  GAN_CKPT=.. sbatch experiments/slurm/exp_cell.slurm` per cell, then
  `aggregate_results.py`.
- **Arm-4 (gan×gan)** may leak ~5–7%; caveat or fix before quoting as headline.